# 01 Memahami data FoodSave

**Tujuan:** mengenali isi data restoran sebelum membuat prediksi limbah makanan dalam kilogram. Satu baris adalah satu observasi, belum tentu satu restoran atau satu hari unik. Dataset diambil dari ZIP yang diberikan pengguna. File `submission.csv` bukan sumber label tambahan; test Kaggle tidak digunakan sebagai holdout penelitian.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/experiment.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/experiment.py').exists(), 'Buka notebook dari folder proyek FoodSave.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown, Code
from src.experiment import OUT, ART, load_data, prepare, run_model, evaluate
pd.set_option('display.max_columns', 40)
pd.set_option('display.precision', 4)
get_ipython().run_line_magic('matplotlib', 'inline')


## Membaca data asli
Kode berikut menampilkan contoh baris, tipe kolom, dan jumlah nilai yang belum terisi. Kolom ID hanya penanda, bukan faktor penyebab limbah.

In [2]:
raw = pd.read_csv(ROOT/'dataset/raw/train.csv')
display(raw.head(10))
display(pd.DataFrame({'tipe': raw.dtypes.astype(str), 'kosong': raw.isna().sum(), 'unik': raw.nunique()}))
display(raw.describe(include='all').T)

,ID,date,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg
0,0,2022-12-19,196,13,27.8873,45.3629,0,0,7.7406,intermediate,dairy,28.9465
1,1,2023-11-21,244,15,10.3179,64.4305,1,0,42.3118,NaN,MeAt,51.5491
2,4,2022-02-01,148,16,27.7143,69.0461,1,0,41.1843,Beginner,MeAt,53.0083
3,5,2023-03-19,157,19,19.1739,46.2928,6,0,41.5435,Beginner,MeAt,48.6215
4,6,2022-07-18,297,10,26.3752,79.7411,0,0,26.5251,Intermediate,MEAT,44.1570
5,7,2023-03-02,241,18,16.8635,79.2859,3,0,11.8349,Intermediate,dairy,27.3937
6,8,2022-04-18,443,16,19.8886,77.3281,0,0,22.8627,Beginner,Vegetables,52.1721
7,9,2023-12-16,416,16,18.5596,75.7865,5,1,34.5994,Intermediate,MeAt,72.0524
8,10,2023-07-07,439,18,24.1110,43.3958,4,0,17.4591,EXPERT,dairy,44.2842
9,11,2023-11-07,267,7,25.4125,89.4052,1,0,23.0674,EXPERT,GRAINS,33.2339


,tipe,kosong,unik
ID,int64,0,911
date,str,0,867
meals_served,int64,0,373
kitchen_staff,int64,0,15
temperature_C,float64,0,892
humidity_percent,float64,0,867
day_of_week,int64,0,7
special_event,int64,0,2
past_waste_kg,float64,0,867
staff_experience,str,164,4


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,911.0,NaN,NaN,NaN,528.3271,305.0728,0.0,266.0,531.0,795.5,1049.0
date,911,867,2022-02-01,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
meals_served,911.0,NaN,NaN,NaN,375.405,502.8127,100.0,211.0,306.0,407.0,4730.0
kitchen_staff,911.0,NaN,NaN,NaN,11.9001,4.2852,5.0,8.0,12.0,15.0,19.0
temperature_C,911.0,NaN,NaN,NaN,22.1893,8.9224,-10.3722,15.6846,22.115,28.8043,60.0
humidity_percent,911.0,NaN,NaN,NaN,60.7613,17.3308,30.1211,46.0178,61.634,75.7879,89.9828
day_of_week,911.0,NaN,NaN,NaN,3.0143,2.0095,0.0,1.0,3.0,5.0,6.0
special_event,911.0,NaN,NaN,NaN,0.0856,0.28,0.0,0.0,0.0,0.0,1.0
past_waste_kg,911.0,NaN,NaN,NaN,27.0157,12.7742,5.0084,16.0914,26.8541,38.1499,49.8037
staff_experience,747,4,Beginner,191,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Memeriksa tanggal dan duplikat
Observasi bertanggal sama ditempatkan pada partisi yang sama. Duplikat persis tanpa ID dibuang untuk seluruh eksperimen agar salinan baris tidak membuat hasil terlalu optimistis. Baris bertarget kosong atau bertanggal tidak valid tidak boleh dilatih.

In [3]:
df = load_data()
audit = json.loads((OUT/'data_audit.json').read_text(encoding='utf-8'))
display(audit)
display(df.groupby('split').agg(jumlah=('date','size'), mulai=('date','min'), akhir=('date','max')))
print('Tanggal unik:', df.date.nunique(), 'Observasi:', len(df))

{'source_sha256': 'c2bedb198bf40cc6793921cdfe0573589923996b3347b3361561b114214b9adf',
 'raw_rows': 911,
 'invalid_date_or_target_removed': 0,
 'exact_duplicates_removed_for_all_models': 0,
 'usable_rows': 911,
 'missing_before': {'ID': 0,
  'date': 0,
  'meals_served': 0,
  'kitchen_staff': 0,
  'temperature_C': 0,
  'humidity_percent': 0,
  'day_of_week': 0,
  'special_event': 0,
  'past_waste_kg': 0,
  'staff_experience': 164,
  'waste_category': 0,
  'food_waste_kg': 0},
 'excluded_features': ['ID', 'waste_category', 'past_waste_kg'],
 'sequence_warning': '7 observasi, bukan 7 hari. Tidak ada ID restoran; urutan temporal tidak membuktikan satu seri restoran.',
 'split': {'test': {'n': 174, 'start': '2024-03-11', 'end': '2024-09-26'},
  'train': {'n': 564, 'start': '2022-01-01', 'end': '2023-08-23'},
  'validation': {'n': 173, 'start': '2023-08-24', 'end': '2024-03-10'}}}

,jumlah,mulai,akhir
split,,,
test,174,2024-03-11,2024-09-26
train,564,2022-01-01,2023-08-23
validation,173,2023-08-24,2024-03-10


Tanggal unik: 867 Observasi: 911


## Fitur yang tersedia saat perencanaan
Model memakai jumlah porsi, staf, suhu, kelembapan, acara khusus, pengalaman staf, dan fitur kalender/rasio. `waste_category` dikeluarkan karena mungkin baru diketahui setelah limbah dihasilkan. `past_waste_kg` dikeluarkan karena asal periode/restorannya belum terverifikasi. `day_of_week` dihitung ulang dari tanggal. Suhu dan kelembapan harus berupa perkiraan yang sudah tersedia saat membuat rencana.

**Batasan:** data ini belum membuktikan hubungan sebab-akibat atau pola satu restoran. Jangan menyebut hasilnya jaminan pengurangan limbah.

## Langkah tambahan 1 Kamus kolom untuk pembaca awam
Kamus ini membedakan angka yang ingin diprediksi, calon input, dan penanda. Nama kolom mengikuti file yang benar-benar tersedia, tanpa menambahkan variabel restoran buatan.

In [4]:
kamus = {
'ID':'Penanda baris; bukan input model.', 'date':'Tanggal observasi; dipakai mengurutkan waktu dan membuat kalender.',
'meals_served':'Jumlah porsi; diasumsikan diketahui saat perencanaan.', 'kitchen_staff':'Jumlah staf dapur.',
'temperature_C':'Suhu Celsius; harus berupa perkiraan jika nilai aktual belum diketahui.',
'humidity_percent':'Kelembapan persen; asumsi ketersediaan sama dengan suhu.',
'day_of_week':'Hari yang tercatat; model menghitung ulang dari tanggal.', 'special_event':'Penanda acara khusus.',
'past_waste_kg':'Limbah sebelumnya; dikeluarkan sampai asal periode/restoran jelas.',
'staff_experience':'Tingkat pengalaman staf, kategori teks.', 'waste_category':'Kategori limbah; dikeluarkan karena mungkin baru diketahui setelah limbah terjadi.',
'food_waste_kg':'Target: jumlah limbah dalam kilogram.'}
display(pd.DataFrame({'kolom':list(kamus),'arti':list(kamus.values())}))


,kolom,arti
0,ID,Penanda baris; bukan input model.
1,date,Tanggal observasi; dipakai mengurutkan waktu d...
2,meals_served,Jumlah porsi; diasumsikan diketahui saat peren...
3,kitchen_staff,Jumlah staf dapur.
4,temperature_C,Suhu Celsius; harus berupa perkiraan jika nila...
5,humidity_percent,Kelembapan persen; asumsi ketersediaan sama de...
6,day_of_week,Hari yang tercatat; model menghitung ulang dar...
7,special_event,Penanda acara khusus.
8,past_waste_kg,Limbah sebelumnya; dikeluarkan sampai asal per...
9,staff_experience,"Tingkat pengalaman staf, kategori teks."


## Langkah tambahan 2 Memeriksa isi file test Kaggle
Nama test pada arsip tidak otomatis berarti holdout independen penelitian. Kode berikut memeriksa tumpang tindih ID dan tanggal. Kita tetap membuat test kronologis dari data berlabel agar metrik dapat dihitung dengan target yang benar. Submission bukan sumber observasi tambahan.

In [5]:
kaggle_test=pd.read_csv(ROOT/'dataset/raw/test.csv')
print('Ukuran train asli:',raw.shape,'| test Kaggle:',kaggle_test.shape)
print('Target tersedia di test Kaggle:', 'food_waste_kg' in kaggle_test)
shared=raw[['ID','date']].merge(kaggle_test[['ID','date']],on=['ID','date']).drop_duplicates()
print('Pasangan ID/tanggal yang juga muncul di train:',len(shared))
if len(shared):print('Ada tumpang tindih; jangan menganggap kedua file sebagai dua sumber independen.')


Ukuran train asli: (911, 12) | test Kaggle: (911, 11)
Target tersedia di test Kaggle: False
Pasangan ID/tanggal yang juga muncul di train: 911
Ada tumpang tindih; jangan menganggap kedua file sebagai dua sumber independen.


## Langkah tambahan 3 Menemukan duplikat dan format angka
Duplikat persis diperiksa tanpa ID karena salinan observasi bisa diberi ID baru. Nilai ekstrem tidak otomatis salah; yang diperiksa terlebih dahulu adalah format dan konteksnya. Konversi angka yang gagal akan menjadi kosong dan nantinya diimputasi dari training.

In [6]:
keys=[c for c in raw if c!='ID']
display(raw.loc[raw.duplicated(keys,keep=False)].head(10))
conversion=[]
for col in ['meals_served','kitchen_staff','temperature_C','humidity_percent']:
    s=pd.to_numeric(raw[col],errors='coerce')
    conversion.append({'kolom':col,'kosong_asli':int(raw[col].isna().sum()),'gagal_konversi_tambahan':int((s.isna() & raw[col].notna()).sum()),'minimum_terbaca':s.min(),'maksimum_terbaca':s.max()})
display(pd.DataFrame(conversion))


,ID,date,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg


,kolom,kosong_asli,gagal_konversi_tambahan,minimum_terbaca,maksimum_terbaca
0,meals_served,0,0,100.0000,4730.0000
1,kitchen_staff,0,0,5.0000,19.0000
2,temperature_C,0,0,-10.3722,60.0000
3,humidity_percent,0,0,30.1211,89.9828


## Langkah tambahan 4 Memeriksa apakah data benar-benar harian
LSTM membutuhkan urutan yang bermakna. Beberapa baris pada tanggal yang sama atau jarak tanggal yang tidak selalu satu hari membatasi klaim temporal. Dataset tidak memiliki ID restoran; kita tidak boleh mengarang identitas restoran untuk membentuk seri.

In [7]:
counts=df.groupby('date').size()
dates=df.date.drop_duplicates().sort_values()
gaps=dates.diff().dt.days.dropna()
display(counts.describe().to_frame('observasi per tanggal'))
display(gaps.value_counts().sort_index().to_frame('frekuensi jarak hari'))
print('Tanggal dengan lebih dari satu observasi:',int((counts>1).sum()))
print('Kolom ID restoran tersedia:',any(c.lower() in ['restaurant_id','restoran_id'] for c in raw.columns))


,observasi per tanggal
count,867.0000
mean,1.0507
std,0.2196
min,1.0000
25%,1.0000
50%,1.0000
75%,1.0000
max,2.0000


,frekuensi jarak hari
date,
1.0,749
2.0,103
3.0,13
5.0,1


Tanggal dengan lebih dari satu observasi: 44
Kolom ID restoran tersedia: False


## Insight data understanding dari hasil aktual

In [8]:
display(Markdown(f'Data berlabel memiliki **{len(raw)} baris awal** dan **{len(df)} observasi setelah audit**. Terdapat **{df.date.nunique()} tanggal unik**. Sebanyak **{int((counts>1).sum())} tanggal memiliki lebih dari satu observasi**. Karena itu, satu baris tidak boleh otomatis disebut satu hari restoran.'))
display(Markdown('Langkah berikutnya adalah EDA training. Jangan menggunakan test untuk memilih fitur, mengubah batas outlier, atau mencoba parameter berulang kali.'))


Data berlabel memiliki **911 baris awal** dan **911 observasi setelah audit**. Terdapat **867 tanggal unik**. Sebanyak **44 tanggal memiliki lebih dari satu observasi**. Karena itu, satu baris tidak boleh otomatis disebut satu hari restoran.

Langkah berikutnya adalah EDA training. Jangan menggunakan test untuk memilih fitur, mengubah batas outlier, atau mencoba parameter berulang kali.